In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:

from models.submodel_manager import SubModelManager

mgr = SubModelManager()
print("## サブモデル分割戦略")
print("")
print("現在の実装: 2分割 (turf / dirt)")
print(f"  VALID_KEYS: {mgr.VALID_KEYS}")
print("")
print("比較対象: 7分割")
print("  turf_sprint, turf_mile, turf_intermediate, turf_long")
print("  dirt_sprint, dirt_mile, dirt_intermediate")
print("")
print("7分割は SubModelManager.add_distance_band_features() で")
print("one-hot 列を追加することで実現可能。")


In [ ]:
print("""
## 2分割の利点

1. サンプル十分性 — 各分割 20,000+ サンプル (要件充足)
2. 過学習リスク低 — データ量が多いほど汎化性能が向上
3. 運用シンプル — モデル管理コストが最小

## 7分割のリスク

1. サンプル不足 — dirt_long 等は 20,000 未満の可能性
2. 過学習 — サンプル不足で train AUC >> test AUC
3. 管理コスト — 7つのモデルセットを個別に管理
""")

In [ ]:
print("""
## 比較手順

1. データを 7キー (surface × distance_bin) で分割
2. 各キーで WinTwoStageModel を学習
3. テスト期間で評価:
   - AUC (P(hit) の予測精度)
   - ROI (ev_threshold 以上のベットの回収率)
   - サンプル数 (20,000 未満は警告)
4. 2分割モデルと同じ評価で比較

期待される結果:
  - 2分割: 安定した性能、全体的に良好な ROI
  - 7分割: turf/dirt で性能向上するキーがあるが、
    サンプル不足のキーで過学習
""")

In [ ]:
print("""
## 結論: 2分割 vs 7分割

2分割が推奨される理由:
  - 全キーでサンプル十分性を確保
  - 距離バンドは特徴量としてモデルに認識させる方が効果的
  - SubModelManager.VALID_KEYS = ["turf", "dirt"]

※ 実データでの検証には TrainingPipelineV5 を使用
""")